# 딥러닝심화 과제6 -  DDPM 실습
#### 컴퓨터공학부 인공지능공학과 20231049 정우제
<hr>

## 목차
<hr>

0. [필요한 라이브러리 불러오기 및 환경설정](#0-필요한-라이브러리-불러오기-및-환경설정)
1. [데이터셋 준비 (CIFAR-10)](#1-데이터셋-준비-(CIFAR-10))
2. [DDPM 모델 구현](#2-실험-2-하이퍼파라미터-최적화)
3. [실험 1: 모델 학습 및 샘플링 결과 시각화](#실험-1:-모델-학습-및-샘플링-결과-시각화)
4. [실험 2: 학습량 증가에 따른 성능 분석](#실험-2:-학습량-증가에-따른-성능-분석)
5. [실험 3: 생성 품질 향상을 위한 EMA 기법 적용](#실험-3:-생성-품질-향상을-위한-EMA-기법-적용)
5. [종합 결과 분석 및 결론](#종합-결과-분석-및-결론)

## 0. 필요한 라이브러리 불러오기 및 환경설정
<hr>

* Library 불러오기

In [ ]:
import random
import imageio  
import numpy as np
from argparse import ArgumentParser
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import einops  
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, ConcatDataset
from torchvision.transforms import Compose, ToTensor, Lambda
from torchvision import datasets, transforms
from torchvision.datasets import CIFAR10
from torchvision.utils import save_image
import subprocess
from IPython.display import Image
import copy

* 시드 고정 및 학습 하이퍼파라미터 설정

In [ ]:
# 재현성을 위한 랜덤 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED) 
    

# 학습 하이퍼파라미터 설정
batch_size = 128
n_epochs = 200   
lr = 0.001
dataset_name = "CIFAR10" 

* device 설정

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    
print(device)

## 1.데이터셋 준비 (CIFAR-10)
<hr>

* 이미지를 [-1, 1] 범위로 정규화

In [ ]:
transform = Compose([
    ToTensor(),
    Lambda(lambda x: (x - 0.5) * 2)]
)

* 생성모델은 train/test 구분이 필요없으므로 train,test 합쳤다.

In [ ]:
# Train 데이터셋
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Test 데이터셋
test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# 데이터셋 합치기
full_dataset = ConcatDataset([train_dataset, test_dataset])

* 전체 데이터셋으로 DataLoader 생성

In [ ]:
loader = DataLoader(full_dataset, batch_size, shuffle=True)

* 이미지 시각화 함수

In [ ]:
def show_images(images, title="", num_images=16):

    if isinstance(images, torch.Tensor):
        images = images.detach().cpu()

    images = (images + 1) / 2
    images = torch.clamp(images, 0, 1) 

    if len(images) > num_images:
        images = images[:num_images]
    
    images = images.numpy()
    
    rows = int(len(images) ** 0.5)
    cols = (len(images) + rows - 1) // rows
    
    plt.figure(figsize=(8, 8))
    
    for idx, image in enumerate(images):
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(np.transpose(image, (1, 2, 0)))
        plt.axis('off')
    
    if title:
        plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

* 테스트

In [ ]:
def show_first_batch(loader):
    batch, _ = next(iter(loader))
    show_images(batch, title="First Batch of CIFAR-10")

* CIFAR-10 데이터 shape 확인 (컬러 이미지라 채널이 3)

In [ ]:
batch = next(iter(loader))
print("Image Shape:", batch[0].shape)
print("Label Shape:", batch[1].shape) 

In [ ]:
show_first_batch(loader)

## 2. DDPM 모델 구현
<hr>

* DDPM 모델 정의
    - 노이즈 스케줄: Linear 또는 Sine 선택 가능

In [ ]:
class MyDDPM(nn.Module):
    def __init__(self, network, n_steps=200, min_beta=10 ** -4, max_beta=0.02, 
                 device=None, image_chw=(3, 32, 32), schedule='linear'):
        super(MyDDPM, self).__init__()
        self.n_steps = n_steps
        self.device = device
        self.image_chw = image_chw
        self.network = network.to(device)
        self.schedule = schedule
        
        
        if schedule == 'linear':
            # Linspace schedule
            self.betas = torch.linspace(min_beta, max_beta, n_steps).to(device)
        elif schedule == 'sine':
            # Sine schedule 
            steps = torch.arange(n_steps + 1, dtype=torch.float32) / n_steps
            alphas_cumprod = torch.cos(((steps + 0.008) / 1.008) * torch.pi * 0.5) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            self.betas = torch.clip(1 - alphas_cumprod[1:] / alphas_cumprod[:-1], 0.0001, 0.9999).to(device)
            
        self.alphas = 1 - self.betas
        self.alpha_bars = torch.tensor([torch.prod(self.alphas[:i + 1]) for i in range(len(self.alphas))]).to(device)
        
    def forward(self, x0, t, eta=None):
        n, c, h, w = x0.shape
        a_bar = self.alpha_bars[t]
        if eta is None:
            eta = torch.randn(n, c, h, w).to(self.device)
        noisy = a_bar.sqrt().reshape(n, 1, 1, 1) * x0 + (1 - a_bar).sqrt().reshape(n, 1, 1, 1) * eta
        return noisy
    
    def backward(self, x, t):
        return self.network(x, t)

* U-Net 구성 요소
    - MyBlock : Conv + Normalization + Activation 기본 블록
    - sinusoidal_embedding: Timestep을 벡터로 변환 
    - MyUNet: 노이즈 예측 네트워크 
    - _make_t: Time embedding을 각 층에 맞게 변환하는 MLP

* U-Net 기본 블록

In [ ]:
class MyBlock(nn.Module):
    def __init__(self, shape, in_c, out_c, kernel_size=3, stride=1, padding=1, activation=None, normalize=True):
        super(MyBlock, self).__init__()
        self.ln = nn.LayerNorm(shape)
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size, stride, padding)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size, stride, padding)
        self.activation = nn.SiLU() if activation is None else activation
        self.normalize = normalize
        
    def forward(self, x):
        out = self.ln(x) if self.normalize else x
        out = self.conv1(out)
        out = self.activation(out)
        out = self.conv2(out)
        out = self.activation(out)
        return out

* sinusoidal_embedding

In [ ]:
def sinusoidal_embedding(n, d):
    embedding = torch.zeros(n, d)
    wk = torch.tensor([1 / 10_000 ** (2 * j / d) for j in range(d)])
    wk = wk.reshape((1, d))
    t = torch.arange(n).reshape((n, 1))
    embedding[:,::2] = torch.sin(t * wk[:,::2])
    embedding[:,1::2] = torch.cos(t * wk[:,::2])
    return embedding

* 노이즈 예측을 위한 U-Net 구조

In [ ]:
class MyUNet(nn.Module):
    def __init__(self, n_steps=1000, time_emb_dim=100):
        super(MyUNet, self).__init__()
        # Sinusoidal embedding
        self.time_embed = nn.Embedding(n_steps, time_emb_dim)
        self.time_embed.weight.data = sinusoidal_embedding(n_steps, time_emb_dim)
        self.time_embed.requires_grad_(False)
        
        # First half 
        self.te1 = self._make_te(time_emb_dim, 3)
        self.b1 = nn.Sequential(
            MyBlock((3, 32, 32), 3, 16),
            MyBlock((16, 32, 32), 16, 16),
            MyBlock((16, 32, 32), 16, 16)
        )
        self.down1 = nn.Conv2d(16, 16, 4, 2, 1)
        
        self.te2 = self._make_te(time_emb_dim, 16)
        self.b2 = nn.Sequential(
            MyBlock((16, 16, 16), 16, 32),
            MyBlock((32, 16, 16), 32, 32),
            MyBlock((32, 16, 16), 32, 32)
        )
        self.down2 = nn.Conv2d(32, 32, 4, 2, 1)
        
        self.te3 = self._make_te(time_emb_dim, 32)
        self.b3 = nn.Sequential(
            MyBlock((32, 8, 8), 32, 64),
            MyBlock((64, 8, 8), 64, 64),
            MyBlock((64, 8, 8), 64, 64)
        )
        self.down3 = nn.Sequential(
            nn.Conv2d(64, 64, 4, 2, 1)
        )
        
        # Bottleneck
        self.te_mid = self._make_te(time_emb_dim, 64)
        self.b_mid = nn.Sequential(
            MyBlock((64, 4, 4), 64, 32),
            MyBlock((32, 4, 4), 32, 32),
            MyBlock((32, 4, 4), 32, 64)
        )
        
        # Second half
        self.up1 = nn.ConvTranspose2d(64, 64, 4, 2, 1)
        
        self.te4 = self._make_te(time_emb_dim, 128)
        self.b4 = nn.Sequential(
            MyBlock((128, 8, 8), 128, 64),
            MyBlock((64, 8, 8), 64, 32),
            MyBlock((32, 8, 8), 32, 32)
        )
        
        self.up2 = nn.ConvTranspose2d(32, 32, 4, 2, 1)
        self.te5 = self._make_te(time_emb_dim, 64)
        self.b5 = nn.Sequential(
            MyBlock((64, 16, 16), 64, 32),
            MyBlock((32, 16, 16), 32, 16),
            MyBlock((16, 16, 16), 16, 16)
        )
        
        self.up3 = nn.ConvTranspose2d(16, 16, 4, 2, 1)
        self.te_out = self._make_te(time_emb_dim, 32)
        self.b_out = nn.Sequential(
            MyBlock((32, 32, 32), 32, 16),
            MyBlock((16, 32, 32), 16, 16),
            MyBlock((16, 32, 32), 16, 16, normalize=False)
        )
        
        self.conv_out = nn.Conv2d(16, 3, 3, 1, 1)
        
    def forward(self, x, t):
        t = self.time_embed(t)
        n = len(x)
        out1 = self.b1(x + self.te1(t).reshape(n, -1, 1, 1))
        out2 = self.b2(self.down1(out1) + self.te2(t).reshape(n, -1, 1, 1))
        out3 = self.b3(self.down2(out2) + self.te3(t).reshape(n, -1, 1, 1))
        
        out_mid = self.b_mid(self.down3(out3) + self.te_mid(t).reshape(n, -1, 1, 1))
        
        out4 = torch.cat((out3, self.up1(out_mid)), dim=1)
        out4 = self.b4(out4 + self.te4(t).reshape(n, -1, 1, 1))
        
        out5 = torch.cat((out2, self.up2(out4)), dim=1)
        out5 = self.b5(out5 + self.te5(t).reshape(n, -1, 1, 1))
        
        out = torch.cat((out1, self.up3(out5)), dim=1)
        out = self.b_out(out + self.te_out(t).reshape(n, -1, 1, 1))
        out = self.conv_out(out)
        
        return out
    
    def _make_te(self, dim_in, dim_out):
        return nn.Sequential(
            nn.Linear(dim_in, dim_out),
            nn.SiLU(),
            nn.Linear(dim_out, dim_out)
        )

## 3. 실험 1: 모델 학습 및 샘플링 결과 시각화
<hr>

* 이미지 시각화 함수

In [ ]:
def show_images(images, title=""):
    images = images.detach().cpu()
    images = (images.clamp(-1, 1) + 1) / 2  # [-1, 1] -> [0, 1]
    images = images.permute(0, 2, 3, 1).numpy()  # (N, C, H, W) -> (N, H, W, C)
    
    fig = plt.figure(figsize=(12, 12))
    n_images = len(images)
    rows = int(np.sqrt(n_images))
    cols = (n_images + rows - 1) // rows
    
    for idx, image in enumerate(images):
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(image)
        plt.axis('off')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

* 순방향 확산 과정 시각화 함수

In [ ]:
def show_forward(ddpm, loader, device):
    for batch in loader:
        imgs = batch[0]
        show_images(imgs[:16], "Original images")
        for percent in [0.25, 0.5, 0.75, 1]:
            show_images(
                ddpm(imgs[:16].to(device),
                     [int(percent * ddpm.n_steps) - 1 for _ in range(16)]),
                f"DDPM Noisy images {int(percent * 100)}%"
            )
        break

* 이미지 생성(샘플링) 함수

In [ ]:
def generate_new_images(ddpm, n_samples=16, device=None, frames_per_gif=100, 
                       gif_name="sampling.gif", c=3, h=32, w=32, use_beta_tilde=False):
    frame_idxs = np.linspace(0, ddpm.n_steps, frames_per_gif).astype(np.uint)
    frames = []
    
    with torch.no_grad():
        if device is None:
            device = ddpm.device
        
        x = torch.randn(n_samples, c, h, w).to(device)
        
        for idx, t in enumerate(list(range(ddpm.n_steps))[::-1]):
            time_tensor = (torch.ones(n_samples, 1) * t).to(device).long()
            eta_theta = ddpm.backward(x, time_tensor)
            
            alpha_t = ddpm.alphas[t]
            alpha_t_bar = ddpm.alpha_bars[t]
            
            x = (1 / alpha_t.sqrt()) * (x - (1 - alpha_t) / (1 - alpha_t_bar).sqrt() * eta_theta)
            
            if t > 0:
                z = torch.randn(n_samples, c, h, w).to(device)
                
                if use_beta_tilde:
                    prev_alpha_t_bar = ddpm.alpha_bars[t-1]
                    beta_t = ddpm.betas[t]
                    beta_tilde_t = ((1 - prev_alpha_t_bar)/(1 - alpha_t_bar)) * beta_t
                    sigma_t = beta_tilde_t.sqrt()
                else:
                    beta_t = ddpm.betas[t]
                    sigma_t = beta_t.sqrt()
                
                x = x + sigma_t * z
            
            if idx in frame_idxs or t == 0:
                normalized = x.clone()
                for i in range(len(normalized)):
                    normalized[i] -= torch.min(normalized[i])
                    normalized[i] *= 255 / torch.max(normalized[i])
                
                frame = einops.rearrange(normalized, "(b1 b2) c h w -> (b1 h) (b2 w) c", b1=int(n_samples ** 0.5))
                frame = frame.cpu().numpy().astype(np.uint8)
                frames.append(frame)
    
          
    with imageio.get_writer(gif_name, mode="I") as writer:
        for idx, frame in enumerate(frames):
            writer.append_data(frame)
            if idx == len(frames) - 1:
                for _ in range(frames_per_gif // 3):
                    writer.append_data(frames[-1])
    
    return x

* 학습루프

In [ ]:
def training_loop(ddpm, loader, n_epochs, optim, device, display=False, store_path="ddpm_model.pt"):
    mse = nn.MSELoss()
    best_loss = float("inf")
    n_steps = ddpm.n_steps
    
    for epoch in tqdm(range(n_epochs), desc=f"Training progress", colour="#00ff00"):
        epoch_loss = 0.0
        for step, batch in enumerate(loader): 
            x0 = batch[0].to(device)
            n = len(x0)
            
            eta = torch.randn_like(x0).to(device)
            t = torch.randint(0, n_steps, (n,)).to(device)
            
            noisy_imgs = ddpm(x0, t, eta)
            eta_theta = ddpm.backward(noisy_imgs, t.reshape(n, -1))
            
            loss = mse(eta_theta, eta)
            
            optim.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(ddpm.parameters(), max_norm=1.0)
            
            optim.step()
            
            epoch_loss += loss.item() * len(x0) / len(loader.dataset)
        
        if display and (epoch + 1) % 10 == 0:
            show_images(generate_new_images(ddpm, n_samples=16, device=device), 
                       f"Images generated at epoch {epoch + 1}")
        
        log_string = f"Loss at epoch {epoch + 1}: {epoch_loss:.5f}"
        
        if best_loss > epoch_loss:
            best_loss = epoch_loss
            torch.save(ddpm.state_dict(), store_path)
            log_string += " --> Best model ever (stored)"
        
        print(log_string)

* 하이퍼파라미터 재설정 및 확인

In [ ]:
n_epochs = 200  
lr = 0.0002
print(f"Device: {device}, Epochs: {n_epochs}")

### 실험 1-1: T=1000, Linear Schedule

* 모델 초기화 및 설정

In [ ]:
n_steps = 1000
schedule = 'linear'
store_path = f"ddpm_T{n_steps}_{schedule}.pt"

ddpm1 = MyDDPM(
    MyUNet(n_steps), 
    n_steps=n_steps, 
    min_beta=10 ** -4, 
    max_beta=0.02, 
    device=device, 
    schedule=schedule 
)

* 확산 과정 시각화

In [ ]:
print(" Forward Process 시각화")
show_forward(ddpm1, loader, device)

* 모델 학습

In [ ]:
print(f" 모델 학습 시작 (Saved to {store_path})")
training_loop(
    ddpm1, 
    loader, 
    n_epochs=n_epochs, 
    optim=Adam(ddpm1.parameters(), lr=lr), 
    device=device, 
    store_path=store_path
)

* 학습된 모델 로드 

In [ ]:
print(f"학습된 모델 로드")
print(store_path)
loaded_model = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model.load_state_dict(torch.load(store_path, map_location=device))
loaded_model.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_beta.gif", use_beta_tilde=False
)
show_images(gen_beta, f"Result: T={n_steps} {schedule} (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde, f"Result: T={n_steps} {schedule} (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_beta_tilde.gif",'rb').read())

### 실험 1-2: T=1000, Sine Schedule

* 모델 초기화 및 설정

In [ ]:
n_steps = 1000
schedule = 'sine'
store_path = f"ddpm_T{n_steps}_{schedule}.pt"

ddpm2 = MyDDPM(
    MyUNet(n_steps), 
    n_steps=n_steps, 
    min_beta=10 ** -4, 
    max_beta=0.02, 
    device=device, 
    schedule=schedule 
)

* 확산 과정 시각화

In [ ]:
print(" Forward Process 시각화")
show_forward(ddpm2, loader, device)

* 모델 학습

In [ ]:
print(f" 모델 학습 시작 (Saved to {store_path})")
training_loop(
    ddpm2, 
    loader, 
    n_epochs=n_epochs, 
    optim=Adam(ddpm2.parameters(), lr=lr), 
    device=device, 
    store_path=store_path
)

* 학습된 모델 로드

In [ ]:
print(f" 학습된 모델 로드")
loaded_model = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model.load_state_dict(torch.load(store_path, map_location=device))
loaded_model.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_beta_tilde.gif",'rb').read())

### 실험 1-3: T=250, Linear Schedule

* 모델 초기화 및 설정

In [ ]:
n_steps = 250
schedule = 'linear'
store_path = f"ddpm_T{n_steps}_{schedule}.pt"

ddpm3 = MyDDPM(
    MyUNet(n_steps), 
    n_steps=n_steps, 
    min_beta=10 ** -4, 
    max_beta=0.02, 
    device=device, 
    schedule=schedule 
)

* 확산 과정 시각화

In [ ]:
print(" Forward Process 시각화")
show_forward(ddpm3, loader, device)

* 모델학습

In [ ]:
print(f" 모델 학습 시작 (Saved to {store_path})")
training_loop(
    ddpm3, 
    loader, 
    n_epochs=n_epochs, 
    optim=Adam(ddpm3.parameters(), lr=lr), 
    device=device, 
    store_path=store_path
)

* 학습된 모델 로드

In [ ]:
print(" 학습된 모델 로드")
loaded_model = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model.load_state_dict(torch.load(store_path, map_location=device))
loaded_model.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_beta_tilde.gif",'rb').read())

### 실험 1-4: T=250, Sine Schedule

* 모델 초기화 및 설정

In [ ]:
n_steps = 250
schedule = 'sine'
store_path = f"ddpm_T{n_steps}_{schedule}.pt"

ddpm4 = MyDDPM(
    MyUNet(n_steps), 
    n_steps=n_steps, 
    min_beta=10 ** -4, 
    max_beta=0.02, 
    device=device, 
    schedule=schedule 
)

* 확산 과정 시각화

In [ ]:
print(" Forward Process 시각화")
show_forward(ddpm4, loader, device)

* 모델학습

In [ ]:
print(f" 모델 학습 시작 (Saved to {store_path})")
training_loop(
    ddpm4, 
    loader, 
    n_epochs=n_epochs, 
    optim=Adam(ddpm4.parameters(), lr=lr), 
    device=device, 
    store_path=store_path
)

* 학습된 모델 로드

In [ ]:
loaded_model = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model.load_state_dict(torch.load(store_path, map_location=device))
loaded_model.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_beta_tilde.gif",'rb').read())

## 4. 실험 2: 학습량  증가에 따른 성능 분석
<hr>

* 하이퍼파라미터 재설정 및 확인

In [ ]:
n_epochs = 1000
lr = 0.0002
print(f"Device: {device}, Epochs: {n_epochs}")

### 실험 2-1 : T=1000, Linear Schedule, epochs = 1000

* 모델 학습

In [ ]:
n_steps = 1000
schedule = 'linear'
store_path = f"ddpm_T{n_steps}_{schedule}_{n_epochs}.pt"

ddpm5 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, min_beta=1e-4, max_beta=0.02, device=device, schedule=schedule)

training_loop(
    ddpm5, 
    loader,  
    n_epochs=n_epochs, 
    optim=Adam(ddpm5.parameters(), lr=lr), 
    device=device, 
    store_path=store_path
)

* 학습된 모델 로드

In [ ]:
loaded_model = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model.load_state_dict(torch.load(store_path, map_location=device))
loaded_model.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} {n_epochs} (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} {n_epochs}(Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_beta_tilde.gif",'rb').read())

### 실험 2-2 : T=1000, Sine Schedule, epochs = 1000

* 모델 학습

In [ ]:
n_steps = 1000
schedule = 'sine'
store_path = f"ddpm_T{n_steps}_{schedule}_{n_epochs}.pt"

ddpm6 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, min_beta=1e-4, max_beta=0.02, device=device, schedule=schedule)

training_loop(
    ddpm6, 
    loader,  
    n_epochs=n_epochs, 
    optim=Adam(ddpm6.parameters(), lr=lr), 
    device=device, 
    store_path=store_path
)

* 학습된 모델 로드

In [ ]:
loaded_model = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model.load_state_dict(torch.load(store_path, map_location=device))
loaded_model.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} {n_epochs} (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule}{n_epochs} (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_beta_tilde.gif",'rb').read())

### 실험 2-3 : T=250, Linear Schedule, epochs = 500

In [ ]:
n_epochs = 500
lr = 0.0002
print(f"Device: {device}, Epochs: {n_epochs}")

* 모델 학습

In [ ]:
n_steps = 250
schedule = 'linear'
store_path = f"ddpm_T{n_steps}_{schedule}_{n_epochs}.pt"

ddpm7 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, min_beta=1e-4, max_beta=0.02, device=device, schedule=schedule)

training_loop(
    ddpm7, 
    loader,  
    n_epochs=n_epochs, 
    optim=Adam(ddpm7.parameters(), lr=lr), 
    device=device, 
    store_path=store_path
)

* 학습된 모델 로드

In [ ]:
loaded_model = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model.load_state_dict(torch.load(store_path, map_location=device))
loaded_model.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} {n_epochs} (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} {n_epochs} (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_beta_tilde.gif",'rb').read())

### 실험 2-4 : T=250, Sine Schedule, epochs = 500

* 모델 학습

In [ ]:
n_steps = 250
schedule = 'sine'
store_path = f"ddpm_T{n_steps}_{schedule}_{n_epochs}.pt"

ddpm8 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, min_beta=1e-4, max_beta=0.02, device=device, schedule=schedule)

training_loop(
    ddpm8, 
    loader,  
    n_epochs=n_epochs, 
    optim=Adam(ddpm8.parameters(), lr=lr), 
    device=device, 
    store_path=store_path
)

* 학습된 모델 로드

In [ ]:
loaded_model = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model.load_state_dict(torch.load(store_path, map_location=device))
loaded_model.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} {n_epochs} (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule}{n_epochs} (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_beta_tilde.gif",'rb').read())

## 5. 실험 3: 생성 품질 향상을 위한 EMA 기법 적용
<hr>

* EMA Class

In [ ]:
class EMA:
    def __init__(self, beta=0.9999):
        super().__init__()
        self.beta = beta
        self.step = 0

    def update_model_average(self, ma_model, current_model):
        with torch.no_grad():
            for current_params, ma_params in zip(current_model.parameters(), ma_model.parameters()):
                old_weight, up_weight = ma_params.data, current_params.data
                ma_params.data = self.update_average(old_weight, up_weight)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1 - self.beta) * new

In [ ]:
import os

In [ ]:
os.makedirs("checkpoints", exist_ok=True)

* Ema를 적용하기 위한 학습루프

In [ ]:
def training_loop_ema(ddpm, ema_model, ema_helper, loader, n_epochs, optim, device, display=False, store_path="ddpm_final_ema.pt"):
    mse = nn.MSELoss()
    n_steps = ddpm.n_steps
    
    pbar = tqdm(range(n_epochs), desc=f"Training EMA", colour="#00ffff")
    
    for epoch in pbar:
        epoch_loss = 0.0
        ddpm.train()
        
        for step, batch in enumerate(loader): 
            x0 = batch[0].to(device)
            n = len(x0)
            
            eta = torch.randn_like(x0).to(device)
            t = torch.randint(0, n_steps, (n,)).to(device)
            
            noisy_imgs = ddpm(x0, t, eta)
            eta_theta = ddpm.backward(noisy_imgs, t.reshape(n, -1))
            loss = mse(eta_theta, eta)
            
            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ddpm.parameters(), max_norm=1.0)
            optim.step()
            
            ema_helper.update_model_average(ema_model, ddpm)
            
            epoch_loss += loss.item() * len(x0) / len(loader.dataset)
        
        pbar.set_postfix({'loss': f"{epoch_loss:.5f}"})
        
        if display and (epoch + 1) % 10 == 0:
            show_images(generate_new_images(ema_model, n_samples=16, device=device), 
                        f"EMA Images generated at epoch {epoch + 1}")
        
        log_string = f"Loss at epoch {epoch + 1}: {epoch_loss:.5f}"
        
        if epoch == n_epochs - 1:
            torch.save(ema_model.state_dict(), store_path)
            log_string += " --> Final EMA model saved"
        
        print(log_string)

### 실험 3-1 : T=1000, Linear Schedule, epochs = 1000 (EMA 적용)

In [ ]:
n_epochs = 1000
n_steps = 1000
schedule = 'linear'
lr = 0.0002
store_path = f"checkpoints/ddpm_T{n_steps}_{schedule}_{n_epochs}_EMA.pt"

In [ ]:
ddpm9 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, min_beta=1e-4, max_beta=0.02, device=device, schedule=schedule)
optimizer = Adam(ddpm9.parameters(), lr=lr)

In [ ]:
model_ema = copy.deepcopy(ddpm9)
for param in model_ema.parameters():
    param.requires_grad = False
ema_helper = EMA(beta=0.9999)

* 모델 학습

In [ ]:
training_loop_ema(
    ddpm=ddpm9,
    ema_model=model_ema,
    ema_helper=ema_helper,
    loader=loader, 
    n_epochs=n_epochs,
    optim=optimizer,
    device=device,
    store_path=store_path 
)

* 학습된 모델 로드

In [ ]:
loaded_model_9 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model_9.load_state_dict(torch.load(store_path, map_location=device))
loaded_model_9.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model_9, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_ema_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} {n_epochs} ema (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_ema_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model_9, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_ema_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} {n_epochs} ema (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_ema_beta_tilde.gif",'rb').read())

### 실험 3-2 : T=1000, Sine Schedule, epochs = 1000(EMA 적용)

In [ ]:
n_epochs = 1000
n_steps = 1000
schedule = 'sine'
lr = 0.0002
store_path = f"ddpm_T{n_steps}_{schedule}_{n_epochs}_FINAL_EMA.pt"

In [ ]:
ddpm10 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, min_beta=1e-4, max_beta=0.02, device=device, schedule=schedule)
optimizer = Adam(ddpm10.parameters(), lr=LR)

In [ ]:
model_ema = copy.deepcopy(ddpm10)
for param in model_ema.parameters():
    param.requires_grad = False
ema_helper = EMA(beta=0.9999)

* 모델 학습

In [ ]:
training_loop_ema(
    ddpm=ddpm10,
    ema_model=model_ema,
    ema_helper=ema_helper,
    loader=loader, 
    n_epochs=n_epochs,
    optim=optimizer,
    device=device,
    store_path=store_path 
)

* 학습된 모델 로드

In [ ]:
loaded_model_10 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model_10.load_state_dict(torch.load(store_path, map_location=device))
loaded_model_10.eval()

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model_10, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_ema_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} {n_epochs} ema (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_ema_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model_10, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_ema_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} {n_epochs} ema (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_ema_beta_tilde.gif",'rb').read())

### 실험 3-3 : T=250, Linear Schedule, epochs = 500(EMA 적용)

In [ ]:
n_epochs = 500
n_steps = 250
schedule = 'linear'
lr = 0.0002
store_path = f"ddpm_T{n_steps}_{schedule}_{n_epochs}_FINAL_EMA.pt"

In [ ]:
ddpm11 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, min_beta=1e-4, max_beta=0.02, device=device, schedule=schedule)
optimizer = Adam(ddpm11.parameters(), lr=lr)

In [ ]:
model_ema = copy.deepcopy(ddpm11)
for param in model_ema.parameters():
    param.requires_grad = False
ema_helper = EMA(beta=0.9999)

* 모델 학습

In [ ]:
training_loop_ema(
    ddpm=ddpm11,
    ema_model=model_ema,
    ema_helper=ema_helper,
    loader=loader, 
    n_epochs=n_epochs,
    optim=optimizer,
    device=device,
    store_path=store_path 
)


* 학습된 모델 로드

In [ ]:
loaded_model_11 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model_11.load_state_dict(torch.load(store_path, map_location=device))
loaded_model_11.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model_10, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_ema_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} {n_epochs} ema (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_ema_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model_10, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_ema_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} {n_epochs} ema (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_ema_beta_tilde.gif",'rb').read())

### 실험 3-4 : T=250, Sine Schedule, epochs = 500

In [ ]:
n_epochs = 500
n_steps = 250
schedule = 'sine'
lr = 0.0002
store_path = f"ddpm_T{n_steps}_{schedule}_{n_epochs}_FINAL_EMA.pt"

In [ ]:
ddpm12 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, min_beta=1e-4, max_beta=0.02, device=device, schedule=schedule)
optimizer = Adam(ddpm12.parameters(), lr=LR)

In [ ]:
model_ema = copy.deepcopy(ddpm12)
for param in model_ema.parameters():
    param.requires_grad = False
ema_helper = EMA(beta=0.9999)

* 모델 학습

In [ ]:
training_loop_ema(
    ddpm=ddpm12,
    ema_model=model_ema,
    ema_helper=ema_helper,
    loader=loader, 
    n_epochs=n_epochs,
    optim=optimizer,
    device=device,
    store_path=store_path
)


* 학습된 모델 로드

In [ ]:
loaded_model_12 = MyDDPM(MyUNet(n_steps), n_steps=n_steps, device=device, schedule=schedule)
loaded_model_12.load_state_dict(torch.load(store_path, map_location=device))
loaded_model_12.eval()

* 이미지 생성(Beta)

In [ ]:
print("   >>> Generating with Beta...")
gen_beta = generate_new_images(
    loaded_model_10, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_ema_beta.gif", use_beta_tilde=False
)
show_images(gen_beta[:16], f"Result: T={n_steps} {schedule} {n_epochs} ema (Beta)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_ema_beta.gif",'rb').read())

* 이미지 생성(Beta Tilde)

In [ ]:
print("   >>> Generating with Beta Tilde...")
gen_tilde = generate_new_images(
    loaded_model_10, n_samples=100, device=device, 
    gif_name=f"T{n_steps}_{schedule}_{n_epochs}_ema_beta_tilde.gif", use_beta_tilde=True
)
show_images(gen_tilde[:16], f"Result: T={n_steps} {schedule} {n_epochs} ema (Beta Tilde)")

* 확산 과정 시각화

In [ ]:
Image(open(f"T{n_steps}_{schedule}_{n_epochs}_ema_beta_tilde.gif",'rb').read())

## 6. 종합 결과 분석 및 결론
<hr>

### 6.1 실험 1: 기본 DDPM 생성 결과 분석

실험1에서는 Timestep(T)과 노이즈 스케줄(Linear/Sine)을 다양하게 조합하여 DDPM 모델의 초기 학습 결과를 분석하였다.

---

#### 6.1.1 T=1000, Linear Schedule

![실험1.2_beta](./HW06_experiment_result/실험1.2_beta.png)
*Figure 6.1: T=1000, Linear Schedule - beta 방식*

![실험1.2_beta_tilde](./HW06_experiment_result/실험1.2_beta_tilde.png)
*Figure 6.2: T=1000, Linear Schedule - beta_tilde 방식*

**Beta 방식 결과 (Figure 6.1)**

파랑, 노랑, 빨강같은 원색들이 마치 색종이를 무작위로 붙여놓은 것처럼 보인다. CIFAR-10에는 분명 개, 고양이, 자동차 같은 구체적인 물체들이 있는데, 생성된 이미지에서는 그런 형태를 전혀 찾아볼 수 없는 것을 볼 수 있다.

**Beta_tilde 방식 결과 (Figure 6.2)**

그나마 beta_tilde 방식이 조금 나아 보인다. 자세히 들여다보니 몇몇 타일에서 동물이나 차량의 윤곽이 희미하게나마 보이는 것 같다. 

==> T=1000과 같은 긴 확산 단계는 초기 학습량이 충분하지 않을 경우 객체 구조를 학습하기 어려운 것으로 보인다.

---

#### 6.1.2 T=1000, Sine Schedule

![실험1.2_beta](./HW06_experiment_result/실험1.2_beta.png)
*Figure 6.3: T=1000, Sine Schedule - beta 방식*

![실험1.2_beta_tilde](./HW06_experiment_result/실험1.2_beta_tilde.png)
*Figure 6.4: T=1000, Sine Schedule - beta_tilde 방식*


* Linear와는 완전히 다른 결과가 나왔음. 보라색, 분홍색, 하늘색 등의 파스텔톤 색상이 부드럽게 섞인 추상적 패턴만 형성되었으며, CIFAR-10의 실제 객체와는 관련성이 거의 없었다.

* Sine 스케줄은 노이즈 변화가 완만하여, 32×32와 같이 작은 해상도에서 필요한 윤곽,경계선 등의 정보를 충분히 유지하지 못해 객체 학습이 어려워진 것으로 해석된다.

---

#### 6.1.3 T=250, Linear Schedule

![실험1.3_beta](./HW06_experiment_result/실험1.3_beta.png)
*Figure 6.5: T=250, Linear Schedule - beta 방식*

![실험1.3_beta_tilde](./HW06_experiment_result/실험1.3_beta_tilde.png)
*Figure 6.6: T=250, Linear Schedule - beta_tilde 방식*

* T를 250으로 줄이자 결과가 크게 개선되었다. 개, 말, 새 등 동물의 형태뿐 아니라 자동차, 비행기 등 차량의 구조도 명확하게 나타났다.
    * 객체와 배경의 구분이 명확해졌다.
    * 색상 또한 CIFAR-10의 실제 데이터와 유사하다.
    * 귀, 다리 등 디테일도 비교적 잘 표현되었다.
* 250 스텝은 초기 학습 단계에서 모델이 충분히 학습할 수 있는 적절한 길이였던 것으로 보인다. 

#### 6.1.4 T=250, Sine Schedule

![실험1.4_beta](./HW06_experiment_result/실험1.4_beta.png)
*Figure 6.7: T=250, Sine Schedule - beta 방식*

![실험1.4_beta_tilde](./HW06_experiment_result/실험1.4_beta_tilde.png)

*Figure 6.8: T=250, Sine Schedule - beta_tilde 방식*


* T를 250으로 줄였으니까 Sine 스케줄도 좋아질 거라고 기대했는데, 개선이 안 된 것을 볼 수 있다.
* 여전히 T=1000 Sine과 비슷하게 흐릿하고 추상적인 색상 블록들만 보인다. 

--- 
* 스케줄 변경만으로는 근본적인 성능 한계를 극복하기 어려우며, Sine 스케줄 자체가 CIFAR-10과 같은 구체적 객체 생성에는 적합하지 않은 것으로 판단된다.
---

### 6.2 실험 2: 학습량 증가에 따른 성능 분석

실험 1에서 T=250, Linear Schedule이 가장 좋은 결과를 보였지만, 여전히 디테일과 일관성이 부족했다. 그래서 실험 2에서는 epoch를 대폭 늘려서 학습을 더 오래 시켜봤다. T=1000은 1000 epoch, T=250은 500 epoch로 학습시켰다.

---

#### 6.2.1 T=1000, Linear Schedule (1000 epoch)

![실험2.1_beta](./HW06_experiment_result/실험2.1_beta.png)
*Figure 6.9: T=1000, Linear Schedule (1000 epoch) - beta 방식*

![실험2.1_beta_tilde](./HW06_experiment_result/실험2.1_beta_tilde.png)
*Figure 6.10: T=1000, Linear Schedule (1000 epoch) - beta_tilde 방식*

* 1000 epoch를 돌렸는데, 실험 1의 초기 결과와 크게 다르지 않았다. 
* 여전히 파란색과 주황색 단색 블록들이 대부분을 차지하고 있다. 
* 일부 타일에서 동물이나 차량의 희미한 형태가 보이긴 하지만, 기대했던 것만큼의 극적인 변화는 없었다.
---
* 학습량 부족의 문제가 아니라, T=1000이라는 설정 자체가 CIFAR-10 특성과 맞지 않는 것으로 보인다.


#### 6.2.2 T=1000, Sine Schedule (1000 epoch)

![실험2.2_beta](./HW06_experiment_result/실험2.2_beta.png)
*Figure 6.11: T=1000, Sine Schedule (1000 epoch) - beta 방식*

![실험2.2_beta_tilde](./HW06_experiment_result/실험2.2_beta_tilde.png)
*Figure 6.12: T=1000, Sine Schedule (1000 epoch) - beta_tilde 방식*


* 1000 epoch를 돌렸는데도 Sine 스케줄은 실험 1과 크게 다르지 않았다. 
* 여전히 주황색, 보라색, 파란색 계열의 파스텔톤 색상들이 부드럽게 섞여 있는 추상적인 패턴만 보이고 명확한 객체의 형태는 거의 찾아볼 수 없다.

---
* 학습량이 문제가 아니라 Sine 스케줄 자체가 구체적인 객체 형태를 학습하는 데 적합하지 않은 것 같다. 
* 아마 점진적이고 부드러운 노이즈 변화가 오히려 객체의 구조적 특징을 흐리게 만드는 게 아닐까 생각된다.

---

#### 6.2.3 T=250, Linear Schedule (500 epoch)

![실험2.3_beta](./HW06_experiment_result/실험2.3_beta.png)
*Figure 6.13: T=250, Linear Schedule (500 epoch) - beta 방식*

![실험2.3_beta_tilde](./HW06_experiment_result/실험2.3_beta_tilde.png)
*Figure 6.14: T=250, Linear Schedule (500 epoch) - beta_tilde 방식*

* 실험 2에서 유일하게 성공적인 결과를 보여준 설정이다. 동물(말, 개, 새 등), 차량(자동차, 트럭, 비행기), 배 등 CIFAR-10의 거의 모든 클래스가 명확하게 식별되는것을 볼 수 있다. 
* 색상도 자연스럽고, 배경과 객체가 확실히 구분된다.

--- 
* 실험 1의 T=250 Linear도 좋았지만, 500 epoch를 돌린 지금이 확실히 더 나음. 
- 디테일이 더 정교해짐 (동물의 털, 차량의 구조 등)
- 색상이 더 정확하고 자연스러움
- 흐릿했던 부분들이 선명해짐
- 샘플 간 품질 편차가 줄어듦

**Beta vs Beta_tilde**

두 방식 모두 거의 비슷한 수준의 높은 품질을 보인다.

--- 
- T=250이라는 적절한 복잡도
- 500 epoch라는 충분한 학습량
- Linear 스케줄의 명확한 노이즈 조절
→ 이 세 가지가 완벽하게 조합된 결과인 것 같음

#### 6.2.4 T=250, Sine Schedule (500 epoch)

![실험2.4_beta](./HW06_experiment_result/실험2.4_beta.png)
*Figure 6.15: T=250, Sine Schedule (500 epoch) - beta 방식*

![실험2.4_beta_tilde](./HW06_experiment_result/실험2.4_beta_tilde.png)
*Figure 6.16: T=250, Sine Schedule (500 epoch) - beta_tilde 방식*


* 학습량 증가로 일부 타일에서는 형태가 약하게 드러났지만, 여전히 추상적인 패턴이 주를 이뤘다. Linear 스케줄과 비교하면 품질 차이가 매우 크게 남아 있다.

* Sine 스케줄은 학습량 증가로도 근본적인 한계를 극복하기 어려운 방식임이 확인되었다.

### 6.3 실험 3: EMA 기법 적용 결과
---

#### 6.3.1 T=1000, Linear Schedule + EMA

![실험3.1_beta](./HW06_experiment_result/실험3.1_beta.png)
*Figure 6.17: T=1000, Linear + EMA - beta 방식*

![실험3.1_beta_tilde](./HW06_experiment_result/실험3.1_beta_tilde.png)
*Figure 6.18: T=1000, Linear + EMA - beta_tilde 방식*

---

#### 6.3.2 T=1000, Sine Schedule + EMA

![실험3.2_beta](./HW06_experiment_result/실험3.2_beta.png)
*Figure 6.19: T=1000, Sine + EMA - beta 방식*

![실험3.2_beta_tilde](./HW06_experiment_result/실험3.2_beta_tilde.png)
*Figure 6.20: T=1000, Sine + EMA - beta_tilde 방식*

---

#### 6.3.3 T=250, Linear Schedule + EMA

![실험3.3_beta](./HW06_experiment_result/실험3.3_beta.png)
*Figure 6.21: T=250, Linear + EMA - beta 방식*

![실험3.3_beta_tilde](./HW06_experiment_result/실험3.3_beta_tilde.png)
*Figure 6.22: T=250, Linear + EMA - beta_tilde 방식*

---

#### 6.3.4 T=250, Sine Schedule + EMA

![실험3.4_beta](./HW06_experiment_result/실험3.4_beta.png)
*Figure 6.23: T=250, Sine + EMA - beta 방식*

![실험3.4_beta_tilde](./HW06_experiment_result/실험3.4_beta_tilde.png)
*Figure 6.24: T=250, Sine + EMA - beta_tilde 방식*

---

#### 6.3.5 실험 3 분석

* EMA 적용 시 생성 품질 향상을 기대하였으나, T=1000 Linear 조합을 제외한 대부분의 설정에서 오히려 특정 색상(녹색·분홍색)으로 균일하게 수렴하는 경향이 나타났다.
* 객체 형태가 약화되었으며 색상 편향이 생기고 전반적인 품질이 실험 2 대비 저하되었다.
---

* 이는 EMA decay 설정 또는 적용 시점이 부적절했을 가능성이 있을 수도 있을거같다.

### 6.4 최종 결론

* 세 차례의 실험을 통해 다음과 같은 최적 설정을 도출하였다.
---
T = 250, Linear Schedule, 500 epoch, EMA 미적용

이 조합이 CIFAR-10의 32×32 소형 이미지에 가장 적합한 설정임을 확인하였다.

---

* T=250이 최적이었던 이유

    * 복잡도(확산 단계)가 데이터 크기와 해상도에 적절함
    * T=1000은 지나치게 복잡하여 학습 이득이 거의 없음
---
* Linear Schedule이 Sine보다 우수했던 이유

    * 직선적이며 단계적인 노이즈 증가 → 객체 윤곽 유지에 유리
    * Sine은 노이즈 변화가 완만하여 구조적 정보가 쉽게 소실됨
---
* EMA 미적용이 더 좋았던 이유

    * EMA가 색상 편향 및 디테일 손실을 유발
    * 저해상도 이미지에서는 EMA smoothing이 오히려 구조 정보 학습을 방해할 수 있음
---